# Webtoon Panel Generation - Colab

Generate the SCBE webtoon panel set on a free Colab GPU.

This notebook expects the prompt pack from `artifacts/webtoon/panel_prompts/`.
Current local pack:
- 38 prompt JSON files
- 305 total panel prompts

Recommended upload flow:
1. Zip `artifacts/webtoon/panel_prompts/` on your local machine.
2. Upload `panel_prompts.zip` into this Colab session.
3. Run the cells top to bottom.
4. Download `six_tongues_panels.zip`.
5. Extract into `kindle-app/www/manhwa/` locally, then rebuild the APK.


In [ ]:
!pip install -q diffusers transformers accelerate safetensors torch

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

PROMPTS_DIR = Path("panel_prompts")
ZIP_NAME = Path("panel_prompts.zip")

if ZIP_NAME.exists() and not PROMPTS_DIR.exists():
    with zipfile.ZipFile(ZIP_NAME, "r") as zf:
        zf.extractall(".")

if not PROMPTS_DIR.exists():
    raise FileNotFoundError(
        "Missing panel_prompts/. Upload panel_prompts.zip or the panel_prompts folder first."
    )

prompt_files = sorted(PROMPTS_DIR.glob("*_prompts.json"))
total_panels = 0
for pf in prompt_files:
    with open(pf, "r", encoding="utf-8") as f:
        data = json.load(f)
    total_panels += len(data.get("panels", []))

print(f"Prompt files: {len(prompt_files)}")
print(f"Total panels: {total_panels}")

In [ ]:
import os
import time
import torch
from diffusers import AutoPipelineForText2Image

OUTPUT_DIR = Path("generated_panels")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Loading SDXL Turbo...")
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=False)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

total_generated = 0
total_skipped = 0
t0_global = time.time()

for pf in prompt_files:
    with open(pf, "r", encoding="utf-8") as f:
        chapter_data = json.load(f)

    ch_id = chapter_data["chapter_id"]
    ch_dir = OUTPUT_DIR / ch_id
    ch_dir.mkdir(exist_ok=True)

    print("\n" + "=" * 52)
    print(f"{ch_id}: {chapter_data.get('title', '')[:60]}")
    print(f"Panels: {len(chapter_data['panels'])}")
    print("=" * 52)

    results = []
    for idx, panel in enumerate(chapter_data["panels"], start=1):
        out_path = ch_dir / f"{panel['id']}.png"

        if out_path.exists():
            print(f"  [{idx}] {panel['id']} - exists, skipping")
            total_skipped += 1
            results.append({"id": panel["id"], "path": str(out_path), "skipped": True})
            continue

        print(
            f"  [{idx}/{len(chapter_data['panels'])}] {panel['id']} ({panel['w']}x{panel['h']})...",
            end=" ",
            flush=True,
        )
        t0 = time.time()

        image = pipe(
            prompt=panel["prompt"],
            width=panel["w"],
            height=panel["h"],
            num_inference_steps=4,
            guidance_scale=0.0,
            generator=torch.Generator("cuda").manual_seed(4000 + total_generated),
        ).images[0]

        image.save(str(out_path))
        elapsed = time.time() - t0
        size_kb = os.path.getsize(out_path) / 1024
        print(f"{elapsed:.1f}s, {size_kb:.0f}KB")

        total_generated += 1
        results.append({
            "id": panel["id"],
            "path": str(out_path),
            "elapsed": round(elapsed, 1),
        })

        torch.cuda.empty_cache()

    with open(ch_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "chapter": ch_id,
                "title": chapter_data.get("title", ""),
                "panels": results,
                "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            },
            f,
            indent=2,
        )

elapsed_total = time.time() - t0_global
print("\n" + "=" * 52)
print("COMPLETE")
print(f"Generated: {total_generated}")
print(f"Skipped: {total_skipped}")
print(f"Total time: {elapsed_total:.0f}s ({elapsed_total / 60:.1f}m)")
print(f"Output dir: {OUTPUT_DIR}")
print("=" * 52)

In [ ]:
archive_path = shutil.make_archive("six_tongues_panels", "zip", OUTPUT_DIR)
print(archive_path)
print(f"Zip size: {os.path.getsize(archive_path) / 1024 / 1024:.1f} MB")

In [ ]:
from google.colab import files
files.download("six_tongues_panels.zip")